# 📥 Notebook 01 — Incremental Loading with Watermarks

**Goal:** Process only *new* records on each pipeline run using a watermark table. Avoid reprocessing data you've already loaded.

> **Run time:** ~5 min

## The Problem with Full Reload
> *Full reload: every run drops and reloads the entire table. Works for 10K rows. Breaks at 100M rows. A bank with 50M transactions can't afford a 4-hour reload every night.*

## Watermark Pattern
```
Run 1: Load TXN where TransactionDate > '2024-12-31'  → 2,000 rows
Run 2: Load TXN where TransactionDate > '2025-01-10'  → 200 rows   (Batch 1)
Run 3: Load TXN where TransactionDate > '2025-01-20'  → 200 rows   (Batch 2)
Run 4: Load TXN where TransactionDate > '2025-01-31'  → 200 rows   (Batch 3)
```

Each run only processes what's new. The watermark stores the last processed date.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

# ── Step 1: Create watermark tracking table ──────────────────────────────────
schema = StructType([
    StructField('pipeline_name',   StringType(),    False),
    StructField('last_watermark',  StringType(),    False),
    StructField('rows_processed',  LongType(),      True),
    StructField('updated_at',      TimestampType(), True),
])

# Initialize watermark table if it doesn't exist
spark.sql("""
    CREATE TABLE IF NOT EXISTS pipeline_watermarks (
        pipeline_name  STRING  NOT NULL,
        last_watermark STRING  NOT NULL,
        rows_processed BIGINT,
        updated_at     TIMESTAMP
    ) USING DELTA
""")

# Seed initial watermark (simulate: initial load covered up to end of 2024)
spark.sql("""
    MERGE INTO pipeline_watermarks t
    USING (SELECT 'transactions_incremental' AS pipeline_name) s
    ON t.pipeline_name = s.pipeline_name
    WHEN NOT MATCHED THEN INSERT VALUES ('transactions_incremental', '2024-12-31', 0, current_timestamp())
""")

spark.table('pipeline_watermarks').show()

## Step 2 — Simulate Batch 1 Arriving (Jan 1-10, 2025)

In [ ]:
def run_incremental_load(batch_number):
    """Run one incremental load — reads watermark, loads new data, updates watermark."""
    print(f'\n=== Running Incremental Load — Batch {batch_number} ===')

    # Read current watermark
    wm_row = spark.sql("SELECT last_watermark FROM pipeline_watermarks WHERE pipeline_name = 'transactions_incremental'").collect()[0]
    last_wm = wm_row['last_watermark']
    print(f'Last watermark: {last_wm}')

    # Load only records from this batch that are newer than the watermark
    df_new = spark.read.option('header','true').option('inferSchema','true') \
        .csv('Files/incremental_transactions.csv') \
        .filter(F.col('BatchNumber') == batch_number) \
        .filter(F.col('TransactionDate') > last_wm)

    new_count = df_new.count()
    print(f'New rows to load: {new_count}')

    if new_count == 0:
        print('Nothing to load — watermark is up to date.')
        return 0

    # Append to fact_transactions
    df_new.drop('BatchNumber') \
        .withColumn('_ingested_at', F.current_timestamp()) \
        .write.format('delta').mode('append').saveAsTable('fact_transactions_incremental')

    # Update watermark to max date seen in this batch
    new_max_date = df_new.agg(F.max('TransactionDate')).collect()[0][0]
    spark.sql(f"""
        UPDATE pipeline_watermarks
        SET last_watermark = '{new_max_date}',
            rows_processed = rows_processed + {new_count},
            updated_at     = current_timestamp()
        WHERE pipeline_name = 'transactions_incremental'
    """)

    print(f'New watermark: {new_max_date}')
    print(f'Total rows loaded this run: {new_count}')
    return new_count

# Run Batch 1
run_incremental_load(batch_number=1)

## Step 3 — Run Batches 2 and 3

In [ ]:
run_incremental_load(batch_number=2)
run_incremental_load(batch_number=3)

## Step 4 — Verify: Watermark History & Row Counts

In [ ]:
%%sql
SELECT pipeline_name, last_watermark, rows_processed, updated_at
FROM pipeline_watermarks

In [ ]:
%%sql
-- Confirm incrementally loaded rows exist
SELECT
    YEAR(TransactionDate)  AS Year,
    MONTH(TransactionDate) AS Month,
    COUNT(*)               AS Rows
FROM fact_transactions_incremental
GROUP BY Year, Month
ORDER BY Year, Month

## Step 5 — Idempotency Test: Re-run Batch 3

In [ ]:
# Re-running the same batch should load 0 rows (watermark already advanced)
rows = run_incremental_load(batch_number=3)
assert rows == 0, 'Idempotency violation: re-running loaded duplicate rows!'
print('\n✅ Idempotency confirmed — no duplicate rows loaded on re-run')